# EdgeGuard · Drive veri ön-hazırlığı

Bu notebook veri indirme yetkisi vermez ve lisans kabulünü otomatikleştirmez. Resmî paketleri Drive'a yerleştirdikten sonra klasör düzenini denetler ve her veri setini tek, SHA-256 bağlı `.tar` dosyasına dönüştürür. Eğitim notebook'u Drive'daki binlerce küçük dosyayı okumak yerine bu paketleri `/content` alanına taşır.

Çekirdek eğitim için yalnız **Cityscapes Fine + BDD100K 10K Semantic + IDD20K Part I/II** gerekir. ACDC ve kapalı external setler model dondurulmadan indirilmez.

In [ ]:
import os
import sys
from pathlib import Path

LOCAL_TEST_MODE = os.environ.get("EDGEGUARD_NOTEBOOK_LOCAL_TEST") == "1"
if LOCAL_TEST_MODE:
    PROJECT_ROOT = Path(os.environ.get("EDGEGUARD_PROJECT_ROOT", Path.cwd())).resolve()
    DRIVE_ROOT = Path(os.environ["EDGEGUARD_TEST_DRIVE_ROOT"]).resolve()
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/edgeguard-road")
    DRIVE_ROOT = Path("/content/drive/MyDrive")

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "rescue/semantic-first"
EXPECTED_PROJECT_COMMIT = ""  # Yayınlanan 40 karakterlik commit ile doldurulması önerilir.
SCIENTIFIC_SOURCE_DATASETS = ["cityscapes", "idd20k"]
PROVISIONAL_ENGINEERING_DATASETS = ["bdd100k"]
OPTIONAL_FINAL_DATASETS = []  # Model/protokol freeze sonrası ör. ["acdc"]
DATASETS_TO_BUNDLE = ["cityscapes", "bdd100k", "idd20k"] + OPTIONAL_FINAL_DATASETS
VERIFY_ARCHIVE_HASHES = True  # Resmî arşivleri bir kez SHA-256/MD5 ile kaydeder.
RUN_ARCHIVE_PREPARATION = False  # Arşivler Drive'a yüklendikten sonra bir kez True yapın.
BDD_SOURCE_PROFILE = "kaggle_mirror"  # Drive'daki bdd100k.zip; yalnız audit/smoke kanıtıdır.
CREATE_BUNDLES = True  # Hazırlanan yerel kökten doğrudan tek Drive tar üretir.
REPLACE_BUNDLES = False  # Yalnız kaynak klasörü bilinçli değiştiyse True yapın.
REUSE_VERIFIED_LEGACY = True  # Mevcut hash-bağlı Cityscapes bundle'ını yeniden kullanır.
REPAIR_STALE_EPHEMERAL_PREPARATION = True  # Yalnız iki sabit /content çalışma kökünü temizler.
DOWNLOAD_LATEST_FAILURE_REPORT = False  # Hata sonrası son hücreyi bununla yeniden çalıştırın.


def persist_bootstrap_failure(notebook, stage, error):
    import json
    import re
    import traceback
    import uuid
    from datetime import datetime, timezone
    from zipfile import ZIP_DEFLATED, ZipFile

    failed_at = datetime.now(timezone.utc)
    failure_id = f"{failed_at.strftime('%Y%m%dT%H%M%S.%fZ')}-{stage}-{uuid.uuid4().hex[:8]}"
    root = DRIVE_ROOT / "EdgeGuard/failures/bootstrap" / failure_id
    root.mkdir(parents=True, exist_ok=False)
    rendered = "".join(traceback.format_exception(type(error), error, error.__traceback__))
    rendered = re.sub(r"(?i)(token|password|secret|api[_-]?key)=\S+", r"\1=<redacted>", rendered)
    payload = {
        "record_type": "edgeguard_colab_bootstrap_failure",
        "failure_id": failure_id,
        "failed_at": failed_at.isoformat(),
        "notebook": notebook,
        "stage": stage,
        "error_type": type(error).__name__,
        "traceback": rendered,
    }
    report = root / "failure.json"
    report.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    package = root / "failure-report.zip"
    with ZipFile(package, "w", compression=ZIP_DEFLATED) as archive:
        archive.write(report, arcname="failure.json")
    print("EDGEGUARD BOOTSTRAP FAILURE:", package)
    return package

In [ ]:
import subprocess

try:
    if LOCAL_TEST_MODE:
        print("LOCAL_TEST_MODE: Drive mount, clone ve paket kurulumu atlandı.")
    elif not (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPOSITORY, str(PROJECT_ROOT)], check=True
        )
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
except BaseException as error:
    persist_bootstrap_failure("EdgeGuard_Data_Preflight_Colab.ipynb", "git-clone-or-update", error)
    raise
if not LOCAL_TEST_MODE:
    if EXPECTED_PROJECT_COMMIT:
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "checkout", EXPECTED_PROJECT_COMMIT], check=True
        )
PROJECT_COMMIT = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from edgeguard.rescue.colab_failures import ColabFailureReporter  # noqa: E402

FAILURE_REPORTER = ColabFailureReporter(
    DRIVE_ROOT / "EdgeGuard/failures/data-preflight" / PROJECT_COMMIT,
    notebook="EdgeGuard_Data_Preflight_Colab.ipynb",
    project_commit=PROJECT_COMMIT,
    context={"branch": BRANCH, "local_test_mode": LOCAL_TEST_MODE},
)
FAILURE_REPORTER.add_diagnostic_root("manifests", DRIVE_ROOT / "EdgeGuard/manifests")
FAILURE_REPORTER.install_ipython_hook()
if not LOCAL_TEST_MODE:
    FAILURE_REPORTER.set_stage("project-install")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)], check=True)

In [ ]:
# Drive klasörlerini oluştur, erişim talimatlarını ve eksikleri tek raporda göster.
import json

from edgeguard.rescue.colab_data import load_colab_data_access

FAILURE_REPORTER.set_stage("drive-inventory-and-hashing")
PREFLIGHT_REPORT = DRIVE_ROOT / "EdgeGuard/manifests/colab-data-inventory.json"
inventory_plan = load_colab_data_access(PROJECT_ROOT / "configs/dataset/colab_data_access_v1.yaml")
inventory_command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
    "--drive-root",
    str(DRIVE_ROOT),
    "--output",
    str(PREFLIGHT_REPORT),
    "inventory",
]
if VERIFY_ARCHIVE_HASHES:
    inventory_command.append("--hash-archives")
subprocess.run(inventory_command, check=True)
inventory = json.loads(PREFLIGHT_REPORT.read_text())
for row in inventory["datasets"]:
    print("\n", row["dataset_id"], "=>", row["state"], "|", row["activation_phase"])
    print("resmî kaynak:", row["official_url"])
    print("işlem:", row["instructions"])
    if row["missing_required_paths"]:
        print("eksik hazır yollar:", row["missing_required_paths"])
    for package in row["packages"]:
        print(
            "paket:",
            package["filename"],
            "Drive'da:",
            package["present"],
            "konum:",
            package["location_profile"],
        )
    for package in row.get("engineering_packages", []):
        print(
            "mühendislik paketi:",
            package["filename"],
            "Drive'da:",
            package["present"],
            "profil:",
            package["source_profile"],
            "bilimsel:",
            package["scientific_eligible"],
        )
    if row.get("legacy_compatibility"):
        print("legacy uyumluluk:", row["legacy_compatibility"])

## Arşivden güvenli hazırlama

Notebook hem yeni `MyDrive/EdgeGuard/archives/<dataset_id>/` düzenini hem de mevcut `MyDrive/EdgeGuard/private_inputs/` klasörünü salt-okunur girdi olarak tanır. Giriş bilgisi, cookie veya geçici indirme URL'sini notebook'a yazmayın. Datasetleri elle açmayın. `RUN_ARCHIVE_PREPARATION=True` olduğunda notebook arşivleri sırayla `/content` alanına kopyalar, hash doğrular, güvenli biçimde hazırlar ve doğrudan tek dosyalı Drive bundle üretir.

Mevcut `private_inputs/bdd100k.zip` Kaggle kaynağıdır. Bundle smoke/plumbing ve veri kataloğu için hazırlanır fakat bilimsel manifest olamaz. Ana bilimsel kaynaklar bu nedenle Cityscapes + IDD20K olarak ayarlanmıştır; resmî iki BDD paketi daha sonra gelirse BDD yeniden ana karşılaştırmaya alınabilir.

IDD polygon JSON etiketleri pinned AutoNUE source-ID sözleşmesiyle maskeye çevrilir; Part II JPG görüntüleri korunur. Native polygon ve source-ID maskeler ayrı kalır.

In [ ]:
# Arşivleri dataset bazında hazırla; büyük ağaçları Drive'a küçük dosyalar hâlinde yazma.
import shutil

from edgeguard.rescue.colab_data import preparation_disk_budget

FAILURE_REPORTER.set_stage("dataset-preparation-and-bundling")
CONTENT_ROOT = Path(os.environ.get("EDGEGUARD_TEST_CONTENT_ROOT", "/content"))
PREPARE_ROOT = CONTENT_ROOT / "edgeguard-prepare"
CACHE_ROOT = CONTENT_ROOT / "edgeguard-archive-cache"
archive_root = DRIVE_ROOT / "EdgeGuard/archives"

if RUN_ARCHIVE_PREPARATION:
    if PREPARE_ROOT.exists() or CACHE_ROOT.exists():
        if not REPAIR_STALE_EPHEMERAL_PREPARATION:
            raise RuntimeError("Stale preparation/cache root found; inspect before retrying")
        for owned in (PREPARE_ROOT, CACHE_ROOT):
            if owned.is_symlink() or owned.name not in {
                "edgeguard-prepare",
                "edgeguard-archive-cache",
            }:
                raise RuntimeError(f"Refusing unsafe preparation cleanup: {owned}")
            if owned.exists():
                shutil.rmtree(owned)
    for dataset in DATASETS_TO_BUNDLE:
        row = next(item for item in inventory["datasets"] if item["dataset_id"] == dataset)
        legacy = row.get("legacy_compatibility") or {}
        if REUSE_VERIFIED_LEGACY and legacy.get("usable_for_training_staging"):
            print(dataset, "mevcut doğrulanmış legacy bundle ile yeniden kullanılacak.")
            continue
        if (
            dataset == "bdd100k"
            and BDD_SOURCE_PROFILE == "kaggle_mirror"
            and row.get("ineligible_smoke_bundle_usable")
        ):
            print(dataset, "mevcut provisional mirror bundle ile yeniden kullanılacak.")
            continue
        if row.get("canonical_bundle_usable") and (
            dataset != "bdd100k" or BDD_SOURCE_PROFILE == "official"
        ):
            print(dataset, "mevcut canonical bundle ile yeniden kullanılacak.")
            continue
        if dataset == "bdd100k" and BDD_SOURCE_PROFILE == "kaggle_mirror":
            engineering = [
                item
                for item in row["engineering_packages"]
                if item["source_profile"] == "kaggle_mirror"
            ]
            sources = [Path(item["path"]) for item in engineering]
        else:
            sources = [Path(item["path"]) for item in row["packages"]]
        missing = [str(path) for path in sources if not path.is_file()]
        if missing:
            raise FileNotFoundError("Missing archives: " + ", ".join(missing))
        budget = preparation_disk_budget(inventory_plan, tuple(sources), CONTENT_ROOT)
        print(dataset, "hazırlık disk kapısı:", budget)
        dataset_cache = CACHE_ROOT / dataset
        dataset_cache.mkdir(parents=True)
        local_archives = []
        for source in sources:
            local = dataset_cache / source.name
            shutil.copyfile(source, local)
            local_archives.append(local)
        prepared = PREPARE_ROOT / dataset
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_dataset.py"),
            "--dataset",
            dataset,
            "--destination",
            str(prepared),
            "--ontology",
            str(PROJECT_ROOT / "configs/dataset/semantic_ontology_v2.yaml"),
        ]
        for archive in local_archives:
            command.extend(["--archive", str(archive)])
        if dataset == "bdd100k":
            command.extend(["--source-profile", BDD_SOURCE_PROFILE])
        subprocess.run(command, check=True)
        if CREATE_BUNDLES:
            bundle = [
                sys.executable,
                str(PROJECT_ROOT / "scripts/prepare_colab_data.py"),
                "--drive-root",
                str(DRIVE_ROOT),
                "bundle",
                "--dataset",
                dataset,
                "--source-root",
                str(prepared),
            ]
            if REPLACE_BUNDLES:
                bundle.append("--replace")
            subprocess.run(bundle, check=True)
        shutil.rmtree(dataset_cache)
        shutil.rmtree(prepared)
    if CACHE_ROOT.is_dir():
        CACHE_ROOT.rmdir()
    if PREPARE_ROOT.is_dir():
        PREPARE_ROOT.rmdir()
    subprocess.run(inventory_command, check=True)
    inventory = json.loads(PREFLIGHT_REPORT.read_text())
else:
    print("RUN_ARCHIVE_PREPARATION=False: arşiv yükleme ve inventory incelemesi bekleniyor.")

In [ ]:
# Eğitim notebook'una geçiş kapısı: bilimsel ve provisional durumlar ayrı raporlanır.
FAILURE_REPORTER.set_stage("preflight-readiness-gate")
bundle_root = DRIVE_ROOT / "EdgeGuard/bundles"
missing = []
for dataset in SCIENTIFIC_SOURCE_DATASETS:
    row = next(item for item in inventory["datasets"] if item["dataset_id"] == dataset)
    legacy = row.get("legacy_compatibility") or {}
    if row.get("canonical_bundle_usable") or legacy.get("usable_for_training_staging"):
        continue
    for suffix in (".prepared.tar", ".prepared.tar.receipt.json"):
        candidate = bundle_root / f"{dataset}{suffix}"
        if not candidate.is_file():
            missing.append(str(candidate))
if missing:
    print("Bilimsel eğitim öncesi eksikler:\n- " + "\n- ".join(missing))
else:
    print("BİLİMSEL VERİ KAPISI GEÇTİ — Cityscapes + IDD20K hazır.")
bdd_row = next(item for item in inventory["datasets"] if item["dataset_id"] == "bdd100k")
print("BDD provisional mirror bundle:", bdd_row.get("ineligible_smoke_bundle_usable", False))
print("BDD resmî bilimsel bundle:", bdd_row.get("canonical_bundle_usable", False))
print("Hata raporu kökü:", FAILURE_REPORTER.output_root)
if DOWNLOAD_LATEST_FAILURE_REPORT:
    latest_failure = FAILURE_REPORTER.latest_package()
    if latest_failure is None:
        raise RuntimeError("İndirilecek hata paketi bulunamadı")
    if not LOCAL_TEST_MODE:
        from google.colab import files

        files.download(str(latest_failure))
    print("Hata paketi:", latest_failure)